# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [24]:
import pandas as pd

# 1. Load the three datasets from raw URLs
url1 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv'
url2 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv'
url3 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv'

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

# 2. Concatenate into a single DataFrame
df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

# 3. Standardize column headers (lowercase & underscores)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 4. Resolve overlapping/duplicate columns caused by header variations
if 'st' in df.columns:
    df['state'] = df['state'].fillna(df['st'])
    df = df.drop(columns=['st'])

gender_cols = df.loc[:, df.columns == 'gender']
if gender_cols.shape[1] > 1:
    df['gender'] = gender_cols.iloc[:, 0].fillna(gender_cols.iloc[:, 1])
    df = df.loc[:, ~df.columns.duplicated()]

# 5. Drop completely empty rows
df = df.dropna(how='all')

# 6. Clean and format 'customer_lifetime_value'
df['customer_lifetime_value'] = df['customer_lifetime_value'].astype(str).str.replace('%', '')
df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')
df['customer_lifetime_value'] = (df['customer_lifetime_value'] / 100).round(2)

# 7. Clean and parse 'number_of_open_complaints' without lambda
def extract_complaint_count(val):
    if pd.isna(val):
        return 0
    val_str = str(val)
    if '/' in val_str:
        return int(val_str.split('/')[1])
    return int(float(val_str))

df['number_of_open_complaints'] = df['number_of_open_complaints'].apply(extract_complaint_count)

# 8. Standardize categorical values in 'gender'
gender_map = {
    'Male': 'M', 'male': 'M', 'M': 'M',
    'Female': 'F', 'femal': 'F', 'Femal': 'F', 'F': 'F'
}
df['gender'] = df['gender'].map(gender_map)

# 9. Standardize state abbreviations
state_map = {
    'AZ': 'Arizona',
    'WA': 'Washington',
    'NV': 'Nevada',
    'CA': 'California',
    'Cali': 'California',
    'OR': 'Oregon'
}
df['state'] = df['state'].replace(state_map)

# 10. Verify final DataFrame info and structure
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9137 entries, 0 to 12073
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9137 non-null   object 
 1   gender                     8985 non-null   object 
 2   education                  9137 non-null   object 
 3   customer_lifetime_value    9130 non-null   float64
 4   income                     9137 non-null   float64
 5   monthly_premium_auto       9137 non-null   float64
 6   number_of_open_complaints  9137 non-null   int64  
 7   policy_type                9137 non-null   object 
 8   vehicle_class              9137 non-null   object 
 9   total_claim_amount         9137 non-null   float64
 10  state                      9137 non-null   object 
dtypes: float64(4), int64(1), object(6)
memory usage: 856.6+ KB


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [25]:
import pandas as pd

# Load marketing customer analysis dataset
url = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv'
df = pd.read_csv(url)

# Display shape and basic info
print("Dataset shape:", df.shape)
df.info()

Dataset shape: (10910, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   unnamed:_0                     10910 non-null  int64  
 1   customer                       10910 non-null  object 
 2   state                          10910 non-null  object 
 3   customer_lifetime_value        10910 non-null  float64
 4   response                       10910 non-null  object 
 5   coverage                       10910 non-null  object 
 6   education                      10910 non-null  object 
 7   effective_to_date              10910 non-null  object 
 8   employmentstatus               10910 non-null  object 
 9   gender                         10910 non-null  object 
 10  income                         10910 non-null  int64  
 11  location_code                  10910 non-null  object 
 12  marital_status     

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [26]:
import pandas as pd

# 1. Ensure sales_channel column and total_claim_amount/total revenue metrics exist
# We sum 'total_claim_amount' or 'customer_lifetime_value' depending on revenue definition, 
# typically 'total_claim_amount' or custom revenue column in this dataset.

# Create pivot table for Sales Channel vs Total Revenue
sales_channel_pivot = pd.pivot_table(
    df, 
    values='total_claim_amount', 
    index='sales_channel', 
    aggfunc='sum'
).round(2)

# Rename column for clarity
sales_channel_pivot.columns = ['total_revenue']
sales_channel_pivot = sales_channel_pivot.sort_values(by='total_revenue', ascending=False)

print(sales_channel_pivot)

               total_revenue
sales_channel               
Agent             1810226.82
Branch            1301204.00
Call Center        926600.82
Web                706600.04
